In [ ]:
# --- LPA repo path bootstrap (added during repository reorganization) ---
# Makes this notebook find the project source (src/) and the data folders at the
# repository root, no matter which notebooks/<topic>/ subfolder it lives in.
# Run this cell first.
import os, sys

_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "pyproject.toml")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root)                       # so relative data paths (e.g. "Dataset/...") resolve
_src = os.path.join(_root, "src")
if _src not in sys.path:
    sys.path.insert(0, _src)          # so `from LPA import ...` works without installing
print("working dir set to repo root:", _root)


## Pre Process Data

In [1]:
import pandas as pd
import re
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

## Load Dataset

In [2]:
both_unseparated = pd.read_csv("Idea-Dataset/dataset/Both-Unseparated.csv")

In [3]:
both_unseparated

,Type,Idea
0,Human,Reducing the amount of bathroom lights active
1,Human,Collaboration with local communities
2,Human,Time and water limited shower heads
3,AI,Collaboration With Local Communities
4,AI,Dining Hall
...,...,...
1256,AI,**Green Building Projects**: Propose retrofitt...
1257,AI,**Green Energy Solutions**: Develop a plan for...
1258,AI,**Eco-Friendly Events**: Organize a series of ...
1259,AI,**Sustainable Dining Initiatives**: Collaborat...


In [4]:
miro = pd.read_csv("Idea-Dataset/dataset/MIRO - All Ideas  (LPA).csv")

In [5]:
miro

,Type,Idea
0,Human,Events
1,Human,Reducing Waste
2,Human,"$5,000 given to each res hall"
3,Human,Seminars
4,Human,Food events
...,...,...
459,AI,**Green Building Projects**: Propose retrofitt...
460,AI,**Green Energy Solutions**: Develop a plan for...
461,AI,**Eco-Friendly Events**: Organize a series of ...
462,AI,**Sustainable Dining Initiatives**: Collaborat...


In [6]:
noda = pd.read_csv("Idea-Dataset/dataset/NODA - All Ideas (LPA).csv")

In [7]:
noda

,Type,Idea
0,Human,Reducing the amount of bathroom lights active
1,Human,Collaboration with local communities
2,Human,Time and water limited shower heads
3,AI,Collaboration With Local Communities
4,AI,Dining Hall
...,...,...
791,Human,Donating Food Med Cost
792,Human,Bow Electric Shuttle
793,Human,Swipe In Technology And Culture Low Cost
794,Human,Donating Food


In [8]:
def append_word_to_type(df, word):
    df['Type'] = df['Type'].apply(lambda x: f"{x}{word}")
    return df


In [9]:
noda1 = pd.read_csv("Idea-Dataset/dataset/NODA - All Ideas (LPA).csv")
miro1 = pd.read_csv("Idea-Dataset/dataset/MIRO - All Ideas  (LPA).csv")

In [10]:
noda1 = filter_non_collaborative (noda1)
miro1 = filter_non_collaborative (miro1)

NameError: name 'filter_non_collaborative' is not defined

In [10]:
noda1= append_word_to_type(noda1 , "noda")

In [11]:
miro1= append_word_to_type(miro1 , "miro")

In [12]:
both_separated = pd.concat([miro1, noda1], ignore_index=True) 

In [13]:
both_separated

,Type,Idea
0,Humanmiro,Events
1,Humanmiro,Reducing Waste
2,Humanmiro,"$5,000 given to each res hall"
3,Humanmiro,Seminars
4,Humanmiro,Food events
...,...,...
1255,Humannoda,Donating Food Med Cost
1256,Humannoda,Bow Electric Shuttle
1257,Humannoda,Swipe In Technology And Culture Low Cost
1258,Humannoda,Donating Food


In [15]:
both_separated.to_csv("Idea-Dataset/dataset/both_separated.csv", index=False)

# Clean Datasets

In [12]:
!pip install textblob --user

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ---------------------------------------- 0.0/624.3 kB ? eta -:--:--
   ------- -------------------------------- 112.6/624.3 kB 3.3 MB/s eta 0:00:01
   ----------- ---------------------------- 174.1/624.3 kB 2.1 MB/s eta 0:00:01
   ------------------- -------------------- 307.2/624.3 kB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 624.3/624.3 kB 3.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ----------------- ---------------------- 0.7/1.5 MB 20.8 MB/s eta 0:00:01
   ------------------------------------ --- 1.4/1.5 MB 14.5 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 16.1 MB/s eta 0:00:00
  Attempting uninstall: nltk
    Found existing installation: nltk 3.8.1
    Uninstalling nltk-3.8.1:
      Successfully uninstalled nltk-3.8.1


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.1 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
!pip install language_tool_python --user

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ---------------------------------------- 0.0/72.5 kB ? eta -:--:--
   ---------------------------------------- 72.5/72.5 kB 2.0 MB/s eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.1 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import pandas as pd
from textblob import TextBlob
import language_tool_python

def correct_spelling_and_grammar(df, column_name):
    """
    Corrects spelling and grammar mistakes in a specified column of a DataFrame.

    Args:
        df (pd.DataFrame): The DataFrame containing text data.
        column_name (str): The column name to be processed.

    Returns:
        pd.DataFrame: The DataFrame with corrected text in the specified column.
    """
    tool = language_tool_python.LanguageTool('en-US')

    def correct_text(text):
        if pd.isnull(text):
            return text  # Return if value is NaN
        
        # Correct spelling using TextBlob
        corrected_text = str(TextBlob(text).correct())
        
        # Correct grammar using LanguageTool
        matches = tool.check(corrected_text)
        corrected_text = language_tool_python.utils.correct(corrected_text, matches)
        
        return corrected_text

    # Apply correction to the specified column
    df[column_name] = df[column_name].apply(correct_text)
    return df


In [14]:
import re
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

def clean_idea_column(df, stemming=False, remove_stopwords=False):
    ps = PorterStemmer()
    stop_words = set(stopwords.words('english'))

    def clean_text(text):
        # Remove Markdown syntax
        text = re.sub(r'\*\*|##|\*|_', '', text)
        # Remove leading parentheses and content inside
        text = re.sub(r'^\([^)]*\)\s*', '', text)
        # Remove punctuation
        text = re.sub(r'[^\w\s]', '', text) 
        # Convert to lowercase
        text = text.lower()
        # Stemming if enabled
        if stemming:
            text = ' '.join(ps.stem(word) for word in text.split())
        # Remove stopwords if enabled
        if remove_stopwords:
            text = ' '.join(word for word in text.split() if word not in stop_words)
        return text
    # df = correct_spelling_and_grammar(df , 'Idea')
    df['Idea'] = df['Idea'].apply(clean_text)
    return df

In [15]:
noda= clean_idea_column(noda,stemming=True, remove_stopwords=True)

In [16]:
noda

,Type,Idea
0,Human,reduc amount bathroom light activ
1,Human,collabor local commun
2,Human,time water limit shower head
3,AI,collabor local commun
4,AI,dine hall
...,...,...
791,Human,donat food med cost
792,Human,bow electr shuttl
793,Human,swipe technolog cultur low cost
794,Human,donat food


In [17]:
miro = clean_idea_column(miro,stemming=True, remove_stopwords=True)

In [18]:
miro

,Type,Idea
0,Human,event
1,Human,reduc wast
2,Human,5000 given hall
3,Human,seminar
4,Human,food event
...,...,...
459,AI,green build project propos retrofit exist buil...
460,AI,green energi solut develop plan instal solar p...
461,AI,ecofriendli event organ seri sustainabilitythe...
462,AI,sustain dine initi collabor dine servi impleme...


In [19]:
both_separated = clean_idea_column(both_separated,stemming=True, remove_stopwords=True)

In [20]:
both_separated

,Type,Idea
0,Humanmiro,event
1,Humanmiro,reduc wast
2,Humanmiro,5000 given hall
3,Humanmiro,seminar
4,Humanmiro,food event
...,...,...
1255,Humannoda,donat food med cost
1256,Humannoda,bow electr shuttl
1257,Humannoda,swipe technolog cultur low cost
1258,Humannoda,donat food


In [21]:
both_unseparated = clean_idea_column(both_unseparated,stemming=True, remove_stopwords=True)

In [22]:
both_unseparated

,Type,Idea
0,Human,reduc amount bathroom light activ
1,Human,collabor local commun
2,Human,time water limit shower head
3,AI,collabor local commun
4,AI,dine hall
...,...,...
1256,AI,green build project propos retrofit exist buil...
1257,AI,green energi solut develop plan instal solar p...
1258,AI,ecofriendli event organ seri sustainabilitythe...
1259,AI,sustain dine initi collabor dine servi impleme...


## Turn CSV to Word Count

In [23]:
from collections import Counter
def word_count_by_type(df):
    word_counts = []
    for doc_type, group in df.groupby('Type'):
        all_words = ' '.join(group['Idea']).split()
        word_freq = Counter(all_words)
        for word, freq in word_freq.items():
            word_counts.append({'document': doc_type, 'element': word, 'frequency_in_document': freq})
    
    return pd.DataFrame(word_counts)


In [24]:
miro_WC = word_count_by_type(miro)

In [25]:
miro_WC

,document,element,frequency_in_document
0,AI,renew,31
1,AI,energi,66
2,AI,project,22
3,AI,invest,7
4,AI,solar,19
...,...,...,...
1048,Human,also,1
1049,Human,lot,1
1050,Human,heatenergi,1
1051,Human,system,1


In [26]:
noda_WC = word_count_by_type(noda)

In [27]:
noda_WC

,document,element,frequency_in_document
0,AI,collabor,11
1,AI,local,7
2,AI,commun,19
3,AI,dine,4
4,AI,hall,4
...,...,...,...
601,Human,benefit,1
602,Human,clg,1
603,Human,govn,1
604,Human,med,1


In [28]:
both_separated_WC = word_count_by_type(both_separated)

In [29]:
both_separated_WC

,document,element,frequency_in_document
0,AImiro,renew,31
1,AImiro,energi,66
2,AImiro,project,22
3,AImiro,invest,7
4,AImiro,solar,19
...,...,...,...
1654,Humannoda,benefit,1
1655,Humannoda,clg,1
1656,Humannoda,govn,1
1657,Humannoda,med,1


In [30]:
both_unseparated_WC = word_count_by_type(both_unseparated)

In [31]:
both_unseparated_WC

,document,element,frequency_in_document
0,AI,collabor,28
1,AI,local,88
2,AI,commun,88
3,AI,dine,31
4,AI,hall,21
...,...,...,...
1324,Human,best,1
1325,Human,probabl,1
1326,Human,also,1
1327,Human,lot,1


## Run Experiment

In [ ]:
!pip install bottleneck --user

In [ ]:
!pip install altair --user

In [34]:
import pandas as pd
import numpy as np

def combine_dataframes_with_spacing(df_list, main_df, output_path):
    """
    Combines DataFrames side by side, including their indices as columns, with a separator column between them.
    
    Parameters:
    df_list (list of pd.DataFrame): List of DataFrames to combine after the main DataFrame.
    main_df (pd.DataFrame): The primary DataFrame to start with.
    output_path (str): Path to save the combined DataFrame (e.g., 'output.csv').
    """
    # Reset index for main_df and convert index to a column
    main_df_reset = main_df.reset_index()
    combined_df = main_df_reset.copy()
    
    # Reset index for each DataFrame in the list
    dfs_reset = [df.reset_index() for df in df_list]
    
    # Add each DataFrame with a separator column
    for df in dfs_reset:
        # Add a separator column filled with NaN
        separator_col = pd.DataFrame({' ': [np.nan] * len(combined_df)})
        combined_df = pd.concat([combined_df, separator_col], axis=1)
        
        # Concatenate the next DataFrame
        combined_df = pd.concat([combined_df, df], axis=1)
    
    # Save the result without adding an extra index column
    combined_df.to_csv(output_path, index=False)
    print(f"Combined DataFrame saved to {output_path}")

In [35]:
from LPA import Corpus, sockpuppet_distance
import altair as alt
from typing import List
import os
def export_signatures(dataframes: List[pd.DataFrame] , outputFolder="LPAresults/signitures/" , index=True):
    for df in dataframes:
        filename = df.name + '-Signature.csv'
        df.to_csv(outputFolder+ filename, index=index)

def run_lpa(freq,exp_name , epsilon_frac =2):
    os.makedirs("Idea-Dataset/Experiments/"+exp_name , exist_ok=True)
    os.makedirs("Idea-Dataset/Experiments/"+exp_name+"/Signatures/" , exist_ok=True)
    os.makedirs("Idea-Dataset/Experiments/"+exp_name+"/Word_Count/" , exist_ok=True)
    corpus = Corpus(freq=freq)
    dvr = corpus.create_dvr()
    epsilon = 1 / (len(dvr) * epsilon_frac)
    dvr.to_csv("Idea-Dataset/Experiments/"+exp_name+"/Signatures/DVR.csv" ,index=True )
    signatures = corpus.create_signatures(epsilon=epsilon, sig_length=20, distance="KLDe")
    export_signatures(signatures , outputFolder ="Idea-Dataset/Experiments/"+exp_name+"/Signatures/")
    freq.to_csv("Idea-Dataset/Experiments/"+exp_name+"/Word_Count/"+exp_name+"-Word-Count.csv" ,index=False)
    combine_dataframes_with_spacing(signatures , dvr , "Idea-Dataset/Experiments/"+exp_name+"/Signatures/summary.csv")


In [36]:
run_lpa(noda_WC , "noda")

Combined DataFrame saved to Idea-Dataset/Experiments/noda/Signatures/summary.csv


In [37]:
run_lpa(miro_WC , "miro")

Combined DataFrame saved to Idea-Dataset/Experiments/miro/Signatures/summary.csv


In [38]:
run_lpa(both_separated_WC, "both_separated")

Combined DataFrame saved to Idea-Dataset/Experiments/both_separated/Signatures/summary.csv


In [39]:
run_lpa(both_unseparated_WC, "both_unseparated")

Combined DataFrame saved to Idea-Dataset/Experiments/both_unseparated/Signatures/summary.csv


## Remove Collab

In [59]:

def filter_non_collaborative(df):
    """
    Remove rows where Type is 'Collaborative' from the dataframe
    
    Args:
        df (pandas.DataFrame): Input dataframe with 'Type' column
        
    Returns:
        pandas.DataFrame: Filtered dataframe
    """
    return df[df['Type'] != 'Collaborative']

In [138]:
miro = filter_non_collaborative(miro)

In [139]:
both_unseparated = filter_non_collaborative(both_unseparated)

In [140]:
noda = filter_non_collaborative(noda)

In [63]:
miro

,Type,Idea
0,Human,Events
1,Human,Reducing Waste
2,Human,"$5,000 given to each res hall"
3,Human,Seminars
4,Human,Food events
...,...,...
459,AI,**Green Building Projects**: Propose retrofitt...
460,AI,**Green Energy Solutions**: Develop a plan for...
461,AI,**Eco-Friendly Events**: Organize a series of ...
462,AI,**Sustainable Dining Initiatives**: Collaborat...


## LDA

In [45]:
import pandas as pd
import altair as alt
import numpy as np
from pathlib import Path
import sys, os
  
from helpers import read
 
import bottleneck as bn
from LPA import Corpus, sockpuppet_distance
from math import floor
from scipy.spatial.distance import cdist, cityblock
import matplotlib.pyplot as plt
from visualize import sockpuppet_matrix, timeline


In [46]:
from copy import deepcopy
from typing import List, Literal
import numpy as np
import altair as alt
import pandas as pd
def timeline(
    df,
    x,
    y,
    title,
    stack: Literal["stack", "center"] | None,
    order: List[str] | None = None,
    name="timeline",
) -> alt.Chart:
    """
    Creates colorful timeline
    Parameters
    ----------
    stack : Literal["stack", "center"] | None
        "stack" for a timeline for zero, including minus values (good for showing distances for instance)
        "center" for a timeline centered aroud the middle of the y axis (good for showing frequencies)
    order : List[str]
        a list of the values in the wanted order (for instance ordering elements with  distance from highest to lowest)
    """
    selection = alt.selection_multi(fields=["element"], bind="legend")
    xx = df[x].drop_duplicates().to_list()
    values = xx[::5] if len(xx) > 100 else xx
    if not order:
        order = df["element"].drop_duplicates().to_list()
    if len(order) > 1:
        color = alt.Color(
            "element",
            scale=alt.Scale(scheme="rainbow"),
            sort=alt.Sort(order),
            legend=alt.Legend(columns=2, labelLimit=1000),
        )
    elif len(order) == 1:
        color = alt.value("blue")
    chart = (
        alt.Chart(df)
        .mark_area()
        .encode(
            x=alt.X(f"{x}:O", title="Date", axis=alt.Axis(values=values)),
            y=alt.Y(
                f"{y}",
                stack=stack,
                title=f"{y.replace('_', ' ').capitalize()}",
            ),
            color=color,
            tooltip=alt.Tooltip(["element", y]),
            opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
        )
        .interactive()
        .properties(width=900)
        .add_selection(selection)
    )
    # chart.save(f"results/{corpus}/timelines/{name}.html")
    return chart


def facet_timeline(
    df,
    x,
    y,
    title,
    stack: Literal["stack", "center"] | None,
    order: List[str] | None = None,
    name="timeline",
    color=None,
    width: int = 900,
) -> alt.Chart:
    """
    Creates colorful timeline
    Parameters
    ----------
    stack : Literal["stack", "center"] | None
        "stack" for a timeline for zero, including minus values (good for showing distances for instance)
        "center" for a timeline centered aroud the middle of the y axis (good for showing frequencies)
    order : List[str]
        a list of the values in the wanted order (for instance ordering elements with  distance from highest to lowest)
    """
    values = (
        list(range(1790, 2022, 5))
        if len(df[x].drop_duplicates()) > 230
        else df[x].tolist()
    )
    if not order:
        order = df["element"].drop_duplicates().to_list()
    if len(order) > 1:
        color = alt.Color(
            "element",
            scale=alt.Scale(scheme="rainbow"),
            sort=alt.Sort(order),
            legend=alt.Legend(columns=2, labelLimit=1000),
        )
        height = {}
    elif len(order) == 1:
        color = alt.value(color)
        height = {"height": 100}
    chart = (
        alt.Chart(df)
        .mark_area()
        .encode(
            x=alt.X(f"{x}:O", title="Date", axis=alt.Axis(values=values)),
            y=alt.Y(f"{y}", stack=stack, axis=alt.Axis(title=None)),
            color=color,
            tooltip=alt.Tooltip(["element", y]),
        )
        .properties(width=width, **height)
    )
    # chart.save(f"results/{corpus}/timelines/{name}.html")
    return chart


In [47]:
def run_LVS(ccdf):
    cc = ccdf.pivot_table(
        index="document", columns="element", values="frequency_in_document"
    ).to_numpy()
    cdist_ = cdist(cc, cc, metric="cityblock")
    cdist_[np.triu_indices(len(cdist_), k=1)] = np.nan
    ccdf = pd.DataFrame(
        cdist_,
        index=ccdf["document"].drop_duplicates(),
        columns=ccdf["document"].drop_duplicates(),
    )
    mmdf = (
        (ccdf / ccdf.max().max())
        .rename_axis(index="year")
        .melt(ignore_index=False, var_name="year ")
        .dropna()
        .reset_index()
    )
    display(sockpuppet_matrix(mmdf))

In [48]:
run_LVS(miro_N.copy())

alt.Chart(...)

In [49]:
miro_WC

,document,element,frequency_in_document
0,AI,renew,31
1,AI,energi,66
2,AI,project,22
3,AI,invest,7
4,AI,solar,19
...,...,...,...
1048,Human,also,1
1049,Human,lot,1
1050,Human,heatenergi,1
1051,Human,system,1


In [50]:
def normalize_frequencies(df):
    """
    Normalize the frequency_in_document column within each document so that 
    the sum of frequencies in each document equals 1.
    
    Parameters:
    df (pd.DataFrame): DataFrame with columns ['element', 'document', 'frequency_in_document']
    
    Returns:
    pd.DataFrame: DataFrame with an additional column 'normalized_frequency'
    """
    df = df.copy()
    df['normalized_frequency'] = df['frequency_in_document'] / df.groupby('document')['frequency_in_document'].transform('sum')
    return df

In [51]:
miro_N = normalize_frequencies(miro_WC)

In [52]:
miro_N

,document,element,frequency_in_document,normalized_frequency
0,AI,renew,31,0.006568
1,AI,energi,66,0.013983
2,AI,project,22,0.004661
3,AI,invest,7,0.001483
4,AI,solar,19,0.004025
...,...,...,...,...
1048,Human,also,1,0.001721
1049,Human,lot,1,0.001721
1050,Human,heatenergi,1,0.001721
1051,Human,system,1,0.001721


In [53]:
def run_LVS(df):
    corpus = Corpus(df)
    dvr = corpus.create_dvr(equally_weighted=True)
    sigs = corpus.create_signatures(distance="JSD")
    sigs[1].to_csv("results/cod/top_30_most_changed.csv")
    ndf = pd.DataFrame(sigs[0])  # Convert to DataFrame
    
    ndf.to_csv("results/cod/distjsd.csv")
    with open('results/cod/list.txt', 'w') as f:
        for item in sigs[0]:
            f.write(f"{item}\n")
    # display(
    #     timeline(
    #         df,
    #         x="document",
    #         y="frequency_in_document",
    #         title="Cause of death, world",
    #         corpus="death_cause",
    #         stack="center",
    #         order=dvr["element"].tolist(),
    #         name=f"1",
    #     )
    # )
     
    spd = sockpuppet_distance(corpus, corpus, heuristic=False)
     
    display(
        sockpuppet_matrix(spd)
        .properties(title="Cause of death, world")
        .configure_axis(title=None)
    )

In [54]:
run_LVS(miro_N)

TypeError: Corpus.create_dvr() got an unexpected keyword argument 'equally_weighted'